# SVM Inference

Loads the linear SVM trained in `train_svm_final.ipynb` to predict probabilities for test soundscape segments and generate `submission.csv`.

Differences from LGBM inference:
- Uses `StandardScaler` for feature scaling.
- Maps 185 active species to 234 submission columns.
- No site-prior boost.

In [1]:
import os, warnings
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import joblib
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.dummy import DummyClassifier
from sklearn.base import BaseEstimator, ClassifierMixin
warnings.filterwarnings('ignore')
print('Imports OK')

Imports OK


In [2]:
DATA_ROOT    = Path('/kaggle/input/competitions/birdclef-2026')
# Paths for SVM artifacts
MODEL_PATH   = '/kaggle/input/notebooks/marialuizalaranjeira/train-svm-optuna-linear/svm_model_v2_linear.pkl'
SCALER_PATH  = '/kaggle/input/notebooks/marialuizalaranjeira/train-svm-optuna-linear/svm_scaler_v2_linear.pkl'
SPECIES_PATH = '/kaggle/input/notebooks/marialuizalaranjeira/train-svm-optuna-linear/svm_active_species_v2_linear.pkl'

SAMPLE_RATE = 32_000
SEGMENT_SEC = 5

for p in (MODEL_PATH, SCALER_PATH, SPECIES_PATH):
    print(f'  {p:<70} exists={Path(p).exists()}')

  /kaggle/input/notebooks/marialuizalaranjeira/train-svm-optuna-linear/svm_model_v2_linear.pkl exists=True
  /kaggle/input/notebooks/marialuizalaranjeira/train-svm-optuna-linear/svm_scaler_v2_linear.pkl exists=True
  /kaggle/input/notebooks/marialuizalaranjeira/train-svm-optuna-linear/svm_active_species_v2_linear.pkl exists=True


In [3]:
# Must be defined before joblib.load can unpickle the model wrapper
class SafeClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, base_estimator):
        self.base_estimator = base_estimator
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        try:
            if len(self.classes_) < 2:
                raise ValueError('only one class')
            from sklearn.base import clone
            self.model_ = clone(self.base_estimator)
            self.model_.fit(X, y)
        except Exception:
            self.model_ = DummyClassifier(strategy='constant', constant=y[0])
            self.model_.fit(X, y)
        return self
    def predict_proba(self, X):
        return self.model_.predict_proba(X)

In [ ]:
# Audio + feature helpers (match training pipeline exactly)
def load_audio_fast(file_path):
    try:
        audio, sr = sf.read(file_path)
        if len(audio.shape) > 1:
            audio = np.mean(audio, axis=1)
        if np.max(np.abs(audio)) == 0:
            audio, sr = librosa.load(file_path, sr=None)
        if sr != SAMPLE_RATE:
            audio = librosa.resample(y=audio, orig_sr=sr, target_sr=SAMPLE_RATE)
        return audio
    except Exception as e:
        print(f'Error loading {file_path}: {e}')
        try:
            audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
            return audio
        except Exception:
            return np.zeros(SAMPLE_RATE * SEGMENT_SEC)

def _stats(arr):
    return [arr.mean(), arr.std(), arr.min(), arr.max()]

def extract_features(y, sr=32_000):
    """~195 features per 5-second window. MUST match training pipeline."""
    target_len = sr * SEGMENT_SEC
    if len(y) > target_len:
        y = y[:target_len]
    else:
        y = np.pad(y, (0, target_len - len(y)), 'constant')

    feat = {}
    S = np.abs(librosa.stft(y, n_fft=1024, hop_length=512))

    mel = librosa.feature.melspectrogram(S=S**2, sr=sr, n_mels=128)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    feat['dominant_mel_band'] = float(np.argmax(np.mean(mel_db, axis=1)))
    feat['mel_mean']          = float(mel_db.mean())
    feat['mel_std']           = float(mel_db.std())

    chunk = sr // 2
    rms_wins = [np.sqrt(np.mean(y[i*chunk:(i+1)*chunk]**2)) for i in range(10)]
    feat['energy_burst_ratio'] = float(np.max(rms_wins) / (np.median(rms_wins) + 1e-6))
    rms = librosa.feature.rms(y=y)[0]
    for k, v in zip(['mean','std','min','max'], _stats(rms)):
        feat[f'rms_{k}'] = float(v)

    mfccs = librosa.feature.mfcc(S=librosa.power_to_db(mel), sr=sr, n_mfcc=20)
    for j in range(20):
        for k, v in zip(['mean','std','min','max'], _stats(mfccs[j])):
            feat[f'mfcc_{j}_{k}'] = float(v)

    delta1 = librosa.feature.delta(mfccs)
    delta2 = librosa.feature.delta(mfccs, order=2)
    for j in range(20):
        feat[f'delta_mfcc_{j}_mean']  = float(delta1[j].mean())
        feat[f'delta_mfcc_{j}_std']   = float(delta1[j].std())
        feat[f'delta2_mfcc_{j}_mean'] = float(delta2[j].mean())
        feat[f'delta2_mfcc_{j}_std']  = float(delta2[j].std())

    centroid = librosa.feature.spectral_centroid(S=S, sr=sr)[0]
    for k, v in zip(['mean','std','min','max'], _stats(centroid)):
        feat[f'centroid_{k}'] = float(v)

    bandwidth = librosa.feature.spectral_bandwidth(S=S, sr=sr)[0]
    for k, v in zip(['mean','std'], _stats(bandwidth)[:2]):
        feat[f'bandwidth_{k}'] = float(v)

    rolloff_85 = librosa.feature.spectral_rolloff(S=S, sr=sr, roll_percent=0.85)[0]
    rolloff_95 = librosa.feature.spectral_rolloff(S=S, sr=sr, roll_percent=0.95)[0]
    feat['rolloff_85_mean'] = float(rolloff_85.mean())
    feat['rolloff_85_std']  = float(rolloff_85.std())
    feat['rolloff_95_mean'] = float(rolloff_95.mean())

    contrast = librosa.feature.spectral_contrast(S=S, sr=sr)
    for band in range(7):
        feat[f'contrast_{band}_mean'] = float(contrast[band].mean())
        feat[f'contrast_{band}_std']  = float(contrast[band].std())

    flatness = librosa.feature.spectral_flatness(S=S)[0]
    feat['flatness_mean'] = float(flatness.mean())
    feat['flatness_std']  = float(flatness.std())

    zcr = librosa.feature.zero_crossing_rate(y)[0]
    for k, v in zip(['mean','std'], _stats(zcr)[:2]):
        feat[f'zcr_{k}'] = float(v)

    return np.array(list(feat.values()), dtype=np.float32)

print('Helpers defined')

Helpers defined


In [ ]:
# Load model + scaler + species list
svm_model      = joblib.load(MODEL_PATH)
scaler         = joblib.load(SCALER_PATH)
ACTIVE_SPECIES = joblib.load(SPECIES_PATH)
print(f'Model loaded. Active species: {len(ACTIVE_SPECIES)}')

# Build submission column order and lookups
sample_sub  = pd.read_csv(DATA_ROOT / 'sample_submission.csv')
ALL_SPECIES = sample_sub.columns[1:].tolist()
N_SPECIES   = len(ALL_SPECIES)
MISSING_PROB = 1.0 / N_SPECIES
UNIFORM      = np.full(N_SPECIES, MISSING_PROB, dtype=np.float32)

active_to_sub_idx = np.array([
    ALL_SPECIES.index(sp) if sp in ALL_SPECIES else -1
    for sp in ACTIVE_SPECIES
], dtype=np.int64)
missing = (active_to_sub_idx < 0).sum()
print(f'Active species not found in submission: {missing} (should be 0)')
print(f'Submission columns: {N_SPECIES}')

Model loaded. Active species: 185
Active species not found in submission: 0 (should be 0)
Submission columns: 234


In [ ]:
# Inference loop
rows_by_stem = {}
for row_id in sample_sub['row_id']:
    parts = row_id.rsplit('_', 1)
    if len(parts) == 2 and parts[1].isdigit():
        rows_by_stem.setdefault(parts[0], []).append((row_id, int(parts[1])))
    else:
        print(f'WARNING unexpected row_id: {row_id}')

print(f'Soundscapes to process: {len(rows_by_stem)}')

results = {}
for stem, segments in tqdm(rows_by_stem.items(), desc='Soundscapes'):
    test_path  = DATA_ROOT / 'test_soundscapes'  / f'{stem}.ogg'
    train_path = DATA_ROOT / 'train_soundscapes' / f'{stem}.ogg'
    y_full = load_audio_fast(str(test_path))
    if np.max(np.abs(y_full)) == 0:
        y_full = load_audio_fast(str(train_path))

    for row_id, end_time in segments:
        try:
            start = (end_time - SEGMENT_SEC) * SAMPLE_RATE
            stop  =  end_time                * SAMPLE_RATE
            y_seg = y_full[start:stop]
            if len(y_seg) == 0 or np.max(np.abs(y_seg)) == 0:
                results[row_id] = UNIFORM.copy()
                continue

            feat        = extract_features(y_seg, sr=SAMPLE_RATE).reshape(1, -1)
            feat_scaled = scaler.transform(feat)
            active_probs = svm_model.predict_proba(feat_scaled)[0]   # shape (N_ACTIVE,)

            # Map probabilities to submission columns
            row = np.full(N_SPECIES, MISSING_PROB, dtype=np.float32)
            row[active_to_sub_idx] = active_probs
            results[row_id] = row

        except Exception as e:
            print(f'WARNING segment {row_id} failed: {e}')
            results[row_id] = UNIFORM.copy()

print(f'Inference complete. Rows produced: {len(results)}')

Soundscapes to process: 1


Soundscapes:   0%|          | 0/1 [00:00<?, ?it/s]

Error loading /kaggle/input/competitions/birdclef-2026/test_soundscapes/BC2026_Test_0001_S05_20250227_010002.ogg: Error opening '/kaggle/input/competitions/birdclef-2026/test_soundscapes/BC2026_Test_0001_S05_20250227_010002.ogg': System error.
Error loading /kaggle/input/competitions/birdclef-2026/train_soundscapes/BC2026_Test_0001_S05_20250227_010002.ogg: Error opening '/kaggle/input/competitions/birdclef-2026/train_soundscapes/BC2026_Test_0001_S05_20250227_010002.ogg': System error.
Inference complete. Rows produced: 3


In [ ]:
# Post-processing Geographic + temporal prior
import re
from functools import lru_cache
PRIOR_LAMBDA = 0.5
PRIOR_ALPHA  = 0.5     # Laplace smoothing

def _parse_stem_meta(stem):
    m = re.search(r'_(S\d+)_(\d{8})_(\d{6})', stem)
    if m:
        return m.group(1), int(m.group(3)[:2])
    return None, None

if PRIOR_LAMBDA > 0:
    sc_path = DATA_ROOT / 'train_soundscapes_labels.csv'
    if not sc_path.exists():
        print('train_soundscapes_labels.csv not found — priors disabled.')
        PRIOR_LAMBDA = 0.0

if PRIOR_LAMBDA > 0:
    sc = pd.read_csv(sc_path)
    sc['_stem'] = sc['filename'].astype(str).str.replace(r'\.ogg$', '', regex=True)
    sc[['_site', '_hour']] = pd.DataFrame(
        sc['_stem'].apply(_parse_stem_meta).tolist(), index=sc.index)
    sc = sc[sc['_site'].notna()].copy()
    sc['_labels'] = sc['primary_label'].fillna('').astype(str).str.split(';')
    sc_exp = sc.explode('_labels')
    sc_exp = sc_exp[sc_exp['_labels'].astype(str).str.len() > 0].copy()
    sc_exp['_label'] = sc_exp['_labels'].astype(str).str.strip()
    sp_to_idx = {sp: i for i, sp in enumerate(ALL_SPECIES)}
    sc_exp = sc_exp[sc_exp['_label'].isin(sp_to_idx)]

    global_counts = sc_exp['_label'].value_counts()
    N_total = len(sc_exp)
    N_sp    = len(ALL_SPECIES)

    def _global_p(sp):
        return (global_counts.get(sp, 0) + PRIOR_ALPHA) / (N_total + PRIOR_ALPHA * N_sp)

    site_sp  = sc_exp.groupby(['_site', '_label']).size().unstack(fill_value=0)
    site_tot = sc_exp.groupby('_site').size()
    hour_sp  = sc_exp.groupby(['_hour', '_label']).size().unstack(fill_value=0)
    hour_tot = sc_exp.groupby('_hour').size()
    print(f'Prior source : {N_total} (segment,label) pairs')
    print(f'  sites      : {list(site_sp.index)}')
    print(f'  hours      : {sorted(hour_sp.index.tolist())}')

    def _site_p(site, sp):
        if site not in site_sp.index: return _global_p(sp)
        c = site_sp.loc[site, sp] if sp in site_sp.columns else 0
        return (c + PRIOR_ALPHA) / (site_tot[site] + PRIOR_ALPHA * N_sp)

    def _hour_p(hour, sp):
        if hour not in hour_sp.index: return _global_p(sp)
        c = hour_sp.loc[hour, sp] if sp in hour_sp.columns else 0
        return (c + PRIOR_ALPHA) / (hour_tot[hour] + PRIOR_ALPHA * N_sp)

    @lru_cache(maxsize=1024)
    def _log_adj(site, hour):
        v = np.zeros(N_SPECIES, dtype=np.float64)
        for i, sp in enumerate(ALL_SPECIES):
            g = _global_p(sp)
            s = _site_p(site, sp) if site else g
            h = _hour_p(hour, sp) if hour is not None else g
            v[i] = (np.log(s + 1e-9) - np.log(g + 1e-9)
                    + np.log(h + 1e-9) - np.log(g + 1e-9))
        return v

    n_adj = 0
    for rid, prob in list(results.items()):
        stem = rid.rsplit('_', 1)[0]
        site, hour = _parse_stem_meta(stem)
        if site is None and hour is None:
            continue
        adj = _log_adj(site, hour)
        results[rid] = np.clip(
            prob * np.exp(PRIOR_LAMBDA * adj).astype(np.float32), 0.0, 1.0
        ).astype(np.float32)
        n_adj += 1
    print(f'Prior applied to {n_adj} rows  (λ={PRIOR_LAMBDA}, α={PRIOR_ALPHA})')
else:
    print('Priors disabled (PRIOR_LAMBDA=0)')

Prior source : 6244 (segment,label) pairs
  sites      : ['S03', 'S08', 'S09', 'S13', 'S15', 'S18', 'S19', 'S22', 'S23']
  hours      : [0, 1, 2, 3, 4, 6, 7, 18, 19, 20, 21, 22, 23]
Prior applied to 3 rows  (λ=0.5, α=0.5)


In [ ]:
# Post-processing Temporal smoothing
SMOOTH_ALPHA = 0.6   # 1.0 = disabled

if SMOOTH_ALPHA < 1.0:
    stems_w = {}
    for rid in results:
        parts = rid.rsplit('_', 1)
        if len(parts) == 2 and parts[1].isdigit():
            stems_w.setdefault(parts[0], []).append((int(parts[1]), rid))
    n_sm = 0
    side = (1.0 - SMOOTH_ALPHA) / 2.0
    for stem, wl in stems_w.items():
        wl.sort()
        n = len(wl)
        if n < 2:
            continue
        P = np.stack([results[r] for _, r in wl])
        S = P.copy()
        for t in range(n):
            if t == 0:
                S[t] = SMOOTH_ALPHA * P[t] + (1 - SMOOTH_ALPHA) * P[t + 1]
            elif t == n - 1:
                S[t] = SMOOTH_ALPHA * P[t] + (1 - SMOOTH_ALPHA) * P[t - 1]
            else:
                S[t] = SMOOTH_ALPHA * P[t] + side * P[t - 1] + side * P[t + 1]
        for i, (_, r) in enumerate(wl):
            results[r] = np.clip(S[i], 0.0, 1.0).astype(np.float32)
            n_sm += 1
    print(f'Smoothed {n_sm} windows (α={SMOOTH_ALPHA})')
else:
    print('Temporal smoothing disabled (α=1.0)')

Smoothed 3 windows (α=0.6)


In [ ]:
# Post-processing: Sonotype mirroring
import re
ENABLE_SONOTYPE_MIRROR = True

if ENABLE_SONOTYPE_MIRROR:
    sp_to_idx = {sp: i for i, sp in enumerate(ALL_SPECIES)}
    groups = {}
    for sp in ALL_SPECIES:
        m = re.match(r'^(.+?)son\d+$', sp)
        if m:
            groups.setdefault(m.group(1), []).append(sp)
    groups = {k: v for k, v in groups.items() if len(v) >= 2}
    group_idxs = [np.array([sp_to_idx[sp] for sp in v], dtype=np.int64)
                  for v in groups.values()]
    n_species_in_groups = sum(len(v) for v in groups.values())
    print(f'Sonotype groups: {len(group_idxs)}  ({n_species_in_groups} species pooled)')

    for rid in results:
        p = results[rid]
        for idxs in group_idxs:
            p[idxs] = p[idxs].max()
        results[rid] = p
    print('Sonotype mirroring applied')
else:
    print('Sonotype mirroring disabled')

Sonotype groups: 1  (25 species pooled)
Sonotype mirroring applied


In [ ]:
# Build and validate submission
prob_matrix = np.vstack([results[rid] for rid in sample_sub['row_id']])
submission  = pd.DataFrame(prob_matrix, columns=ALL_SPECIES)
submission.insert(0, 'row_id', sample_sub['row_id'].values)

assert list(submission.columns) == list(sample_sub.columns), 'column mismatch'
assert len(submission) == len(sample_sub), 'row count mismatch'
assert not submission.isnull().any().any(), 'NaN found'
assert (submission.iloc[:, 1:].values >= 0).all(), 'negative probability'
assert (submission.iloc[:, 1:].values <= 1).all(), 'probability > 1'

submission.to_csv('submission.csv', index=False)
print(f'submission.csv saved — shape: {submission.shape}')
submission.head(3)

submission.csv saved — shape: (3, 235)


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Test_0001_S05_20250227_010002_5,0.014068,0.001932,0.014068,0.005531,0.014068,0.014068,0.014068,0.014068,0.008178,...,0.014068,0.000884,0.014068,0.014068,0.014068,0.014068,0.014068,0.014068,0.014068,0.014068
1,BC2026_Test_0001_S05_20250227_010002_10,0.014068,0.001932,0.014068,0.005531,0.014068,0.014068,0.014068,0.014068,0.008178,...,0.014068,0.000884,0.014068,0.014068,0.014068,0.014068,0.014068,0.014068,0.014068,0.014068
2,BC2026_Test_0001_S05_20250227_010002_15,0.014068,0.001932,0.014068,0.005531,0.014068,0.014068,0.014068,0.014068,0.008178,...,0.014068,0.000884,0.014068,0.014068,0.014068,0.014068,0.014068,0.014068,0.014068,0.014068
